[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module5/07-feature-engineering.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module5/07-feature-engineering.ipynb)

# Module 5 — Lesson 7: Feature Engineering

**Module:** 5 — Machine Learning Foundations | **Time:** 45 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Choose the correct encoder for ordinal and nominal categorical variables
- Compare the effect of standard, min-max, and robust scaling on data with outliers
- Apply filter, wrapper, and embedded feature selection methods
- Use PCA for dimensionality reduction and interpret explained variance
- Visualise high-dimensional data in 2-D after PCA transformation

In [ ]:
!pip install -q category_encoders

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_wine, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   OrdinalEncoder, OneHotEncoder)
from sklearn.feature_selection import (SelectKBest, f_classif, mutual_info_classif,
                                        SelectFromModel, RFE)
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import category_encoders as ce

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Libraries loaded. category_encoders installed.')

## 1. Encoding Categorical Variables

### When to use each encoder:

| Encoder | Category type | Notes |
|---|---|---|
| `OrdinalEncoder` | Ordered (small, medium, large) | Preserves ordinal relationship |
| `OneHotEncoder` | Nominal (colour, city) | No false ordering; creates k binary columns |
| `TargetEncoder` | High-cardinality nominal | Replaces category with target mean; risk of leakage — use only in pipeline |

In [ ]:
# Create a dataset with mixed categorical types
np.random.seed(42)
n = 300
df = pd.DataFrame({
    'size':    np.random.choice(['small', 'medium', 'large'], n),        # ordinal
    'color':   np.random.choice(['red', 'blue', 'green', 'yellow'], n),  # nominal
    'city':    np.random.choice([f'city_{i}' for i in range(20)], n),    # high cardinality
    'value':   np.random.normal(5, 1, n)
})
y_enc = (df['value'] > 5).astype(int).values
df = df.drop('value', axis=1)

print('Sample data:')
print(df.head())
print(f'\nShape: {df.shape}')
print(f'Unique cities: {df["city"].nunique()}')

# OrdinalEncoder
ord_enc = OrdinalEncoder(categories=[['small', 'medium', 'large']])
size_encoded = ord_enc.fit_transform(df[['size']])
print('\nOrdinalEncoder output (first 5):', size_encoded[:5].ravel())

# OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)
color_encoded = ohe.fit_transform(df[['color']])
print('\nOneHotEncoder features:', ohe.get_feature_names_out())
print('First row:', color_encoded[0])

# TargetEncoder (category_encoders)
te = ce.TargetEncoder(cols=['city'])
city_encoded = te.fit_transform(df[['city']], y_enc)
print('\nTargetEncoder — city column (first 5):')
print(city_encoded.head())

## 2. Encoder Performance Comparison

Different encoders can have a meaningful impact on model accuracy, especially for high-cardinality features.

In [ ]:
# Build a numeric dataset with one high-cardinality categorical feature
X_num = np.random.randn(n, 4)
df_full = pd.DataFrame(np.column_stack([X_num, df['city']]),
                       columns=['f1','f2','f3','f4','city'])
X_num_arr = X_num

results_enc = {}

# OneHotEncoder
ohe_full = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_ohe = np.column_stack([X_num_arr, ohe_full.fit_transform(df[['city']])])
pipe_ohe = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=500))])
results_enc['OneHotEncoder'] = cross_val_score(pipe_ohe, X_ohe, y_enc, cv=5).mean()

# TargetEncoder
df_te = df[['city']].copy()
te2 = ce.TargetEncoder(cols=['city'], smoothing=5)
X_te_arr = np.column_stack([X_num_arr, te2.fit_transform(df_te, y_enc).values])
pipe_te = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=500))])
results_enc['TargetEncoder'] = cross_val_score(pipe_te, X_te_arr, y_enc, cv=5).mean()

# OrdinalEncoder (naive baseline)
ord2 = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_ord = np.column_stack([X_num_arr, ord2.fit_transform(df[['city']])])
pipe_ord = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=500))])
results_enc['OrdinalEncoder'] = cross_val_score(pipe_ord, X_ord, y_enc, cv=5).mean()

print('CV Accuracy by Encoding Method (city column):')
for name, score in sorted(results_enc.items(), key=lambda x: -x[1]):
    print(f'  {name:20s}: {score:.4f}')

## 3. Scaling — Effect on Outliers

Real data often contains outliers. The choice of scaler determines how much influence outliers have on the scaled representation.

In [ ]:
data_bc = load_breast_cancer()
X_bc, y_bc = data_bc.data, data_bc.target

# Inject extreme outliers into feature 0
X_outlier = X_bc.copy()
X_outlier[:5, 0] = X_bc[:, 0].max() * 50

scalers_compare = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
    'RobustScaler':   RobustScaler()
}

X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(X_outlier, y_bc, test_size=0.2,
                                                        stratify=y_bc, random_state=42)

print('Feature 0 — effect of outliers on scaled values:')
for name, sc in scalers_compare.items():
    X_sc_tr = sc.fit_transform(X_tr_bc)
    outlier_val = X_sc_tr[:5, 0].mean()
    normal_val  = X_sc_tr[5:, 0].mean()
    print(f'  {name:20s}  outlier mean: {outlier_val:8.2f}  normal mean: {normal_val:.4f}')

# Model accuracy comparison across scalers
print('\nModel accuracy with outliers present:')
X_tr_o, X_te_o, y_tr_o, y_te_o = train_test_split(X_outlier, y_bc, test_size=0.2,
                                                    stratify=y_bc, random_state=42)
for name, sc in scalers_compare.items():
    pipe = Pipeline([('sc', sc), ('clf', LogisticRegression(max_iter=1000))])
    pipe.fit(X_tr_o, y_tr_o)
    acc = accuracy_score(y_te_o, pipe.predict(X_te_o))
    print(f'  {name:20s}: {acc:.4f}')

## 4. Feature Selection

Reducing the number of features can:
- Improve model accuracy (remove noise)
- Speed up training
- Improve interpretability

Three major strategies:
- **Filter** — rank features by statistical test (SelectKBest)
- **Wrapper** — recursively eliminate features with a model (RFE)
- **Embedded** — regularisation drives coefficients to zero (SelectFromModel + Lasso)

In [ ]:
# Use breast cancer dataset — 30 features
X_bc_clean, y_bc_clean = load_breast_cancer(return_X_y=True)
X_tr_fs, X_te_fs, y_tr_fs, y_te_fs = train_test_split(X_bc_clean, y_bc_clean,
                                                        test_size=0.2, stratify=y_bc_clean, random_state=42)
feature_names = load_breast_cancer().feature_names

# --- Filter: SelectKBest with f_classif ---
skb = SelectKBest(score_func=f_classif, k=10)
skb.fit(X_tr_fs, y_tr_fs)
selected_filter = feature_names[skb.get_support()]
print('SelectKBest (F-score) top 10 features:')
for f, s in sorted(zip(selected_filter, skb.scores_[skb.get_support()]),
                   key=lambda x: -x[1]):
    print(f'  {f:40s} F={s:.1f}')

# --- Wrapper: RFE ---
lr_rfe = LogisticRegression(max_iter=1000)
rfe = RFE(estimator=lr_rfe, n_features_to_select=10, step=1)
rfe.fit(StandardScaler().fit_transform(X_tr_fs), y_tr_fs)
selected_rfe = feature_names[rfe.support_]
print(f'\nRFE (Logistic Regression) top 10: {list(selected_rfe)}')

# --- Embedded: SelectFromModel + Lasso ---
lasso_pipe = Pipeline([('sc', StandardScaler()), ('lasso', Lasso(alpha=0.01, max_iter=5000))])
lasso_pipe.fit(X_tr_fs, y_tr_fs)
sfm = SelectFromModel(lasso_pipe.named_steps['lasso'], prefit=True, threshold=1e-5)
X_lasso_support = sfm.get_support()
print(f'\nLasso SelectFromModel features selected: {X_lasso_support.sum()}')
print([f for f, s in zip(feature_names, X_lasso_support) if s])

In [ ]:
# Compare accuracy: all features vs top 10 from each method
from sklearn.svm import SVC

def eval_features(X_tr, X_te, y_tr, y_te, label):
    pipe = Pipeline([('sc', StandardScaler()), ('clf', SVC(kernel='rbf', probability=True))])
    pipe.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, pipe.predict(X_te))
    print(f'  {label:35s}: {acc:.4f}  ({X_tr.shape[1]} features)')

print('Accuracy comparison:')
eval_features(X_tr_fs, X_te_fs, y_tr_fs, y_te_fs, 'All 30 features')
eval_features(X_tr_fs[:, skb.get_support()], X_te_fs[:, skb.get_support()],
              y_tr_fs, y_te_fs, 'SelectKBest top 10')
eval_features(StandardScaler().fit_transform(X_tr_fs)[:, rfe.support_],
              StandardScaler().fit_transform(X_te_fs)[:, rfe.support_],
              y_tr_fs, y_te_fs, 'RFE top 10')
eval_features(X_tr_fs[:, X_lasso_support], X_te_fs[:, X_lasso_support],
              y_tr_fs, y_te_fs, f'Lasso Embedded ({X_lasso_support.sum()} features)')

## 5. PCA — Principal Component Analysis

PCA finds the directions (principal components) of maximum variance in the data. It reduces dimensionality while retaining as much information as possible.

Key parameters:
- `n_components` — number of components to keep
- `explained_variance_ratio_` — fraction of total variance explained by each component
- **Scree plot** — visualises the explained variance per component

In [ ]:
# Load Wine dataset (13 features, 3 classes)
wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)

# Fit full PCA
pca_full = PCA()
pca_full.fit(X_wine)

evr = pca_full.explained_variance_ratio_
cumsum_evr = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree plot
axes[0].bar(range(1, len(evr)+1), evr, color='steelblue', alpha=0.8, label='Individual')
axes[0].plot(range(1, len(evr)+1), cumsum_evr, 'o-', color='tomato', label='Cumulative')
axes[0].axhline(0.95, linestyle='--', color='green', label='95% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Scree Plot — Wine Dataset')
axes[0].legend()

# 2-D projection
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_wine)
scatter = axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=wine.target, cmap='tab10', s=40)
plt.colorbar(scatter, ax=axes[1], label='Wine class')
axes[1].set_xlabel(f'PC1 ({evr[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({evr[1]*100:.1f}% var)')
axes[1].set_title('Wine Dataset — 2-D PCA Projection')

plt.tight_layout()
plt.show()

n95 = np.argmax(cumsum_evr >= 0.95) + 1
print(f'Components needed to explain 95% variance: {n95}')
print(f'PC1 explains: {evr[0]*100:.1f}%')
print(f'PC2 explains: {evr[1]*100:.1f}%')
print(f'Combined (PC1+PC2): {cumsum_evr[1]*100:.1f}%')

## 6. PCA — Effect on Model Performance

In [ ]:
X_w, y_w = load_wine(return_X_y=True)
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(X_w, y_w, test_size=0.2, stratify=y_w, random_state=42)

results_pca = []
for n_comp in [2, 4, 6, 8, 10, 13]:
    pipe_pca = Pipeline([
        ('sc', StandardScaler()),
        ('pca', PCA(n_components=n_comp)),
        ('clf', LogisticRegression(max_iter=1000))
    ])
    cv_score = cross_val_score(pipe_pca, X_tr_w, y_tr_w, cv=5, scoring='accuracy').mean()
    results_pca.append({'n_components': n_comp, 'cv_accuracy': round(cv_score, 4)})

pca_df = pd.DataFrame(results_pca)
print('PCA Components vs CV Accuracy:')
print(pca_df.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.plot(pca_df['n_components'], pca_df['cv_accuracy'], 'o-', color='steelblue', lw=2)
plt.xlabel('Number of PCA Components')
plt.ylabel('CV Accuracy')
plt.title('PCA Dimensionality Reduction vs Model Accuracy')
plt.xticks(pca_df['n_components'])
plt.tight_layout()
plt.show()

## Practice Exercises

**Exercise 1 — Feature Importance from Trees**
Load `load_wine()`. Fit a `RandomForestClassifier(n_estimators=200)`. Plot the feature importances as a horizontal bar chart sorted from most to least important. Then train a second model using only the top 5 features. Compare accuracy (5-fold CV) between the full and reduced models.

**Exercise 2 — PCA Visualisation of High-Dimensional Data**
Generate a dataset with `make_classification(n_samples=500, n_features=50, n_informative=10)`. Apply StandardScaler then PCA. Plot cumulative explained variance. How many components are needed for 90% variance? Then project to 2-D and visualise the class separation with a scatter plot coloured by class label.

**Exercise 3 — Full Feature Engineering Pipeline**
Using the Wine dataset, build a single `Pipeline` that: (1) scales with `StandardScaler`, (2) applies `SelectKBest(k=8)`, (3) applies `PCA(n_components=4)`, (4) classifies with `LogisticRegression`. Use `GridSearchCV` to tune `selectkbest__k` in `[5, 8, 10, 13]` and `pca__n_components` in `[2, 4, 6]`. Report the best pipeline and its test accuracy.